# Colab setup (run once per session)

Mounts Drive, clones GitHub, extracts training data, installs deps.

**Option A — S3 (recommended):** Store AWS credentials in Colab Secrets (🔑 left sidebar) and run the S3 sync cell below. No zip needed.

**Option B — Drive zip (fallback):** upload zip from work PC (`scripts/sync_to_cloud.ps1`) to `My Drive/id-forensics/id_forensics_home_data.zip`, then skip the S3 cell.

Then open a training notebook (`colab_train_stage2_screen.ipynb`, etc.).

In [ ]:
# Only secret: GitHub token for private repo (leave "" if public)
GITHUB_TOKEN = ""  # paste token here for private repo only — never commit

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/id-forensics-model")
BOOT = REPO / "scripts" / "colab_bootstrap.py"
user, repo = "ansisvaisla", "id-forensics-model"
url = (
    f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
    if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
)

os.chdir("/content")
if REPO.is_dir() and not BOOT.is_file():
    print("Removing stale clone (missing bootstrap)...")
    subprocess.run(["rm", "-rf", str(REPO)], check=True)

if not REPO.is_dir():
    print("Cloning repo...")
    subprocess.run(["git", "clone", url, str(REPO)], check=True)

if not BOOT.is_file():
    raise FileNotFoundError(
        "colab_bootstrap.py not on GitHub yet. On work PC run:\n"
        "  git push origin main\n"
        "Then re-run this cell."
    )

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
cb.setup_workspace(github_token=GITHUB_TOKEN)

## Option A — Pull images directly from S3 (recommended)

Requires four Colab Secrets (🔑 sidebar): `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `S3_BUCKET_NAME`.

Skip this cell if you already extracted data from the Drive zip above.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/id-forensics-model/scripts")
import colab_bootstrap as cb
importlib.reload(cb)

# Downloads raw images + label export from S3, then re-runs convert + split.
# Only new/changed files are downloaded (size-based skip).
cb.download_from_s3(rebuild_splits=True, run_convert=True)